# 4. Complex derivatives, Taylor error, and the sensitivity surrogate

Read [Chapters 6–7](../06-sensitivity.md). This notebook evaluates a fixed
four-gate device; it does **not** optimize or use held-out experimental data.
We use `prepare_problem`, its actual `sensitivity` objective, the implicit
response solve, centered finite differences, and full perturbed solves.
Reserved optimization seed fields are saved but unused; only the listed
educational seed generates draws.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import sys
import numpy as np
import os, tempfile
if "MPLCONFIGDIR" not in os.environ:
    _mpl_cache=tempfile.TemporaryDirectory(prefix="qwave-teaching-mpl-")
    os.environ["MPLCONFIGDIR"]=_mpl_cache.name
import matplotlib.pyplot as plt

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "pyproject.toml").is_file() and (p / "src/qwave").is_dir())
sys.path.insert(0, str(ROOT / "docs/derivations"))
from verification import VerificationBundle
plt.rcParams.update({"font.size": 10, "axes.grid": True, "grid.alpha": .2})

model_config=json.loads((ROOT/"examples/optimize_quick.json").read_text())
model_config["model"]["scattering"].update(grid_points=161,energy_points=81,energy_min_mev=20.,energy_max_mev=60.)
model_config["model"]["disorder"].update(retained_modes=4,boundary_taper_nm=3.)
model_config["optimizer"]["initialization_seed"]=740101
model_config["ensembles"].update(training_seed=740102,validation_seed=740103,heldout_seed=740104)
CONFIG={"seeds":[740001],"optimization_setup":model_config,"linear_draws":1024,"full_solve_draws":8,
 "mode_steps_mev_sqrt_nm":[1.,.5,.25,.01,1e-5,1e-9,1e-30],"disorder_scales":[0.,.02,.04,.08,.5,1.,4.],
 "synthetic_global_phase_rad":.8,"synthetic_phase_coefficient":.7,"synthetic_dimensionless_variance":.01,
 "approximations":["fixed four-gate design, no search", "spatially weighted retained KL modes",
 "real centered coefficient perturbations, fixed leads", "paired full/retained draws",
 "first-order predictions are not assumed unitary", "uniform-energy RMS in Taylor plots; packet weighting in J"],
 "criteria":{"derivative_best_atol":1e-7,"refinement_ratio_bounds":[2.5,5.5],
 "covariance_contraction_rtol":1e-11,"linear_mean_standard_errors":5.,"conservation_atol":1e-10}}
output=Path(globals().get("RUN_OUTPUT",ROOT/"results"/("learning-04-"+datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ"))))
bundle=VerificationBundle(output,CONFIG,scope="fixed-design mathematical verification; synthetic metric counterexample; no optimizer ranking")
print(json.dumps(CONFIG,indent=2))


In [ ]:
from qwave.optimization import prepare_problem
from qwave.optimization_config import OptimizeConfig
from qwave.sensitivity import derivative_study,validate_linearization
from qwave.disorder import sample_disorder
from qwave.wavepackets import analyze_packet
problem=prepare_problem(OptimizeConfig.from_mapping(model_config))
device=problem.device
u=device.gates.amplitudes_mev
potential=problem.potential(u)
cache=problem.cache()
clean,response=cache.solve(potential,response=True)
J,J_per_sigma2,D=problem.sensitivity(response)
phi=device.embedded_shapes
lam=device.modes.retained_eigenvalues
G=response.dt_dpotential_per_mev
cov=(phi*lam)@phi.T
nodal=np.einsum("ei,ij,ej->e",G,cov,G.conj())
energy_weight=problem.packet.weights_mev*problem.importance_density
J_nodal=float(np.dot(energy_weight,nodal).real)
study=derivative_study(device.x_nm,potential,device.energy_mev,phi,CONFIG["mode_steps_mev_sqrt_nm"],**problem.arguments)
fd_error=np.max(abs(study.finite_difference-study.linear_response),axis=(1,2))
bundle.table("finite-difference",[{"h_mev_sqrt_nm":h,"max_derivative_error_per_mev_sqrt_nm":err,
 "unresolved_modes":int(np.sum(flag))} for h,err,flag in zip(study.steps_mev_sqrt_nm,fd_error,study.unresolved_perturbations)])
bundle.json("derivative-mode-diagnostics",study.mode_rows)
bundle.check("covariance contraction equals actual objective",abs(J_nodal-J)<1e-11*max(J,1e-15),
 measured={"J_actual":J,"J_nodal":J_nodal,"max_imaginary_remainder":float(max(abs(nodal.imag)))},
 criterion="Weighted modal objective and G C_K G-dagger contraction agree to relative 1e-11")
bundle.check("complex finite differences agree with implicit response",float(min(fd_error))<1e-7,
 measured=fd_error,criterion="At least one declared step resolves every retained derivative within 1e-7 (meV sqrt(nm))^-1")
bundle.check("under-roundoff perturbation reported",np.any(study.unresolved_perturbations[-1]),
 measured=study.unresolved_perturbations[-1],criterion="The 1e-30 step must not be accepted as evidence of zero sensitivity")
samples=sample_disorder(device.modes,sample_count=CONFIG["linear_draws"],seed=CONFIG["seeds"][0])
eta=samples.coefficients_mev_sqrt_nm[:,:len(lam)]
delta_t=eta@D.T
linear_norms=(abs(delta_t)**2)@energy_weight
J_mean=float(np.mean(linear_norms));J_se=float(np.std(linear_norms,ddof=1)/np.sqrt(len(linear_norms)))
bundle.check("expected linear norm agrees with J",abs(J_mean-J)<5*J_se,
 measured={"J":J,"sample_mean":J_mean,"sample_standard_error":J_se},criterion="1024 Gaussian coefficient draws; mean within 5 estimated standard errors, not a held-out result")
n=CONFIG["full_solve_draws"]
full_potentials=np.zeros((n,len(device.x_nm)))
full_potentials[:,device.channel_indices]=samples.full_mev[:n]
linearization=validate_linearization(device.x_nm,potential,device.energy_mev,phi,eta[:n],CONFIG["disorder_scales"],
 clean.t,D,full_disorder_mev=full_potentials,**problem.arguments)
bundle.table("linearization",linearization.rows)
small_errors=np.array([row["rms_complex_error"] for row in linearization.rows[1:4]])
ratios=small_errors[1:]/small_errors[:-1]
bundle.check("small-disorder quadratic Taylor error",np.all((ratios>2.5)&(ratios<5.5)),
 measured=ratios,criterion="Doubling the first three positive scales gives error ratios 2.5 to 5.5 for this fixed design")
bundle.check("perturbed full solves conserve probability",linearization.max_conservation_error<1e-10,
 measured=linearization.max_conservation_error,criterion="All physical full solves conserve T+R within 1e-10; Taylor polynomials are not clipped or required to do so")


The Taylor-error RMS uses the validator's normalized **uniform energy** measure,
while $J$ above uses the configured packet density. They are not equated.
The table records KL truncation error separately and retains cases where the
linearized probability exceeds one.

For the metric counterexample, multiply the physical clean output by a
synthetic constant global phase. Chapter 7's family $e^{ic\zeta}t_0(E)$ has
dimensionless $c,\zeta$, variance $s_\zeta^2$, and a positive linear surrogate
$c^2s_\zeta^2\int w|t_0|^2dE$. Its conditional fidelity remains one exactly.
This is an identity check, not a claim about realizable gate responses.

In [ ]:
synthetic=analyze_packet(device.energy_mev,problem.packet.amplitude,
                        clean.t*np.exp(1j*CONFIG["synthetic_global_phase_rad"]),clean.t)
J_phase=float(CONFIG["synthetic_phase_coefficient"]**2*CONFIG["synthetic_dimensionless_variance"]*
              np.dot(energy_weight,abs(clean.t)**2))
bundle.check("global-phase surrogate counterexample",J_phase>0 and abs(synthetic.raw_fidelity-1)<1e-12,
 measured={"synthetic_surrogate":J_phase,**synthetic.summary()},criterion="Positive synthetic linear surrogate while conditional fidelity remains one; no transport-design claim")
residuals,_=problem.residuals(u,problem.packet_metrics(clean.t),problem.config.constraints["minimum_clean_transmission_values"][0])
bundle.json("constraint-residuals",residuals)
bundle.json("objective",{"J_dimensionless":J,"J_over_sigma2_per_mev2":J_per_sigma2,"weighting":problem.config.objective["weighting"],
 "clean_metrics_vs_configured_target":problem.packet_metrics(clean.t),"synthetic_metrics_vs_clean":synthetic.summary(),
 "synthetic_phase_J":J_phase,"response_cache_cost":cache.counters(),"setup_cost":problem.setup_cost})
bundle.json("kl-diagnostics",device.modes.diagnostics)
bundle.arrays("sensitivity",units={"x_nm":"nm","energy_mev":"meV","potential_mev":"meV","gate_contributions":"meV",
 "eigenvalues":"meV^2 nm","phi":"nm^-1/2","G":"meV^-1 complex","D":"(meV sqrt(nm))^-1 complex",
 "finite_difference":"(meV sqrt(nm))^-1 complex","t_plus":"1 complex","t_minus":"1 complex","t0":"1 complex",
 "eta":"meV sqrt(nm)","standard_normals":"1","full_disorder":"meV","full_t":"1 complex","linear_t":"1 complex",
 "full_covariance_t":"1 complex","incoming_A":"meV^-1/2 complex","importance_density":"meV^-1","energy_weights":"meV"},
 x_nm=device.x_nm,energy_mev=device.energy_mev,potential_mev=potential,gate_contributions=device.gates.contributions_mev,
 eigenvalues=lam,phi=phi,G=G,D=D,finite_difference=study.finite_difference,t_plus=study.t_plus,t_minus=study.t_minus,t0=clean.t,
 eta=eta,standard_normals=samples.standard_normals,full_disorder=full_potentials,full_t=linearization.full_t,
 linear_t=linearization.linear_t,full_covariance_t=linearization.full_covariance_t,
 incoming_A=problem.packet.amplitude,importance_density=problem.importance_density,energy_weights=problem.packet.weights_mev)
print(json.dumps({"J":J,"J_over_sigma2_per_mev2":J_per_sigma2,"finite_difference_errors":fd_error.tolist(),
 "linearization":linearization.rows},indent=2))
fig,axs=plt.subplots(2,2,figsize=(10,7.3),layout="constrained")
im=axs[0,0].pcolormesh(device.x_nm,device.energy_mev,abs(G),shading="auto")
axs[0,0].set(xlabel="Position (nm)",ylabel="Energy (meV)");fig.colorbar(im,ax=axs[0,0],label="Nodal |dt/dV_j| (1/meV)")
for k in range(len(lam)):
    axs[0,1].plot(device.energy_mev,lam[k]*abs(D[:,k])**2,label=f"Mode {k+1}")
axs[0,1].set(xlabel="Energy (meV)",ylabel="Linear amplitude variance contribution (1)");axs[0,1].legend()
# Plot all declared steps, including the unresolved extreme, without pretending it converges.
axs[1,0].loglog(study.steps_mev_sqrt_nm,fd_error,"o-")
axs[1,0].set(xlabel="Real coefficient step (meV sqrt(nm))",ylabel="Max derivative error (1/(meV sqrt(nm)))")
scales=np.asarray(CONFIG["disorder_scales"])[1:]
axs[1,1].loglog(scales,[row["rms_complex_error"] for row in linearization.rows[1:]],"o-",label="Taylor error, retained draws")
axs[1,1].loglog(scales,[row["kl_truncation_rms_error"] for row in linearization.rows[1:]],"s-",label="KL truncation, full solves")
axs[1,1].set(xlabel="Disorder strength multiplier (1)",ylabel="Complex t RMS error (1)");axs[1,1].legend(fontsize=8)
bundle.figure(fig,"sensitivity-objective","Fixed-gate physical-model response. Derivatives preserve complex phase; modal variance includes lambda once. Unresolved small steps and unfavorable Taylor/truncation errors remain visible. Taylor RMS is uniform-energy weighted, unlike packet-weighted J.")
fig,axs=plt.subplots(1,2,figsize=(10,3.8),layout="constrained")
for j in range(len(u)):
    axs[0].plot(device.x_nm,device.gates.contributions_mev[:,j],label=f"Gate {j+1}")
axs[0].plot(device.x_nm,potential,"k--",label="Sum + background")
axs[0].set(xlabel="Position (nm)",ylabel="Potential energy (meV)");axs[0].legend(fontsize=8)
integrand=problem.importance_density*np.sum(lam*abs(D)**2,axis=1)
axs[1].plot(device.energy_mev,integrand)
axs[1].fill_between(device.energy_mev,integrand,alpha=.2)
axs[1].set(xlabel="Energy (meV)",ylabel="Sensitivity objective integrand (1/meV)")
bundle.figure(fig,"gates-objective","Fixed bounded four-gate contributions and packet-weighted objective integrand. Its energy integral is dimensionless J, not a fidelity. Clean constraint residuals are saved; no search is performed.")
bundle.finish()
